In [35]:
import torch

In [36]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [37]:
import pandas as pd

In [38]:
torch.manual_seed(0)

In [39]:
np.random.seed(42)

In [40]:
from pathlib import Path

csv_path = Path("student_exam_scores.csv")
if not csv_path.exists():
    csv_path = Path("task/ANN/student_exam_scores.csv")

df = pd.read_csv(csv_path)
df.head()

,student_id,hours_studied,sleep_hours,attendance_percent,previous_scores,exam_score
0,S001,8.0,8.8,72.1,45,30.2
1,S002,1.3,8.6,60.7,55,25.0
2,S003,4.0,8.2,73.7,86,35.8
3,S004,3.5,4.8,95.1,66,34.0
4,S005,9.1,6.4,89.8,71,40.3


In [41]:
df['exam_score'].max()

np.float64(51.3)

In [42]:
X = df[["hours_studied", "sleep_hours", "attendance_percent", "previous_scores"]].values
y = df["exam_score"].values

In [43]:
y = y.astype(np.float32)

In [44]:
print(f"Total samples: {len(X)}")
print(f"Target min score: {y.min():.2f}")
print(f"Target max score: {y.max():.2f}")
print(f"Target mean score: {y.mean():.2f}")

Total samples: 200
Target min score: 17.10
Target max score: 51.30
Target mean score: 33.96


In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
scaler = StandardScaler()

In [63]:
X_train = scaler.fit_transform(X_train)

In [64]:
X_test = scaler.transform(X_test)

In [65]:
X_train_tensor = torch.FloatTensor(X_train)

In [66]:
X_test_tensor = torch.FloatTensor(X_test)

In [67]:
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

In [52]:
print(f"\nTensor shapes:")
print(f"X_train: {X_train_tensor.shape}")
print(f"y_train: {y_train_tensor.shape}")


Tensor shapes:
X_train: torch.Size([160, 4])
y_train: torch.Size([160, 1])


In [76]:
model = nn.Sequential(
    nn.Linear(in_features=4, out_features=32),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(in_features=32, out_features=16),
    nn.ReLU(),
    nn.Linear(in_features=16, out_features=1)
)

print("\n" + "="*50)
print("MODEL ARCHITECTURE")
print("="*50)
print(model)



MODEL ARCHITECTURE
Sequential(
  (0): Linear(in_features=4, out_features=32, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Linear(in_features=16, out_features=1, bias=True)
)


In [54]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

Total trainable parameters: 705


In [69]:
criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [70]:
num_epochs = 100


batch_size = 32



In [71]:
train_losses = []
test_losses  = []

print("\n" + "="*50)
print("TRAINING")
print("="*50)

for epoch in range(num_epochs):
    model.train()

    epoch_loss = 0.0
    batch_count = 0

    num_samples_train = X_train_tensor.size(0)

    for i in range(0, num_samples_train, batch_size):

        X_batch = X_train_tensor[i : i + batch_size]
        y_batch = y_train_tensor[i : i + batch_size]

        optimizer.zero_grad(set_to_none=True)

        y_pred = model(X_batch)

        loss = criterion(y_pred, y_batch)
        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    avg_train_loss = epoch_loss / batch_count
    train_losses.append(avg_train_loss)

    model.eval()

    with torch.no_grad():
        y_test_pred = model(X_test_tensor)
        test_loss   = criterion(y_test_pred, y_test_tensor)
        test_losses.append(test_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"Train MSE: {avg_train_loss:.4f} | "
              f"Test MSE:  {test_loss.item():.4f}")



TRAINING
Epoch [ 20/300] | Train Loss: 0.0669 | Test Loss: 0.0942 | LR: 0.003000
Epoch [ 40/300] | Train Loss: 0.0539 | Test Loss: 0.0937 | LR: 0.001500
Early stopping at epoch 49
Best validation loss restored: 0.0871


In [ ]:
print("\n" + "="*50)
print("EVALUATION")
print("="*50)

model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor)

y_true = y_test_tensor.numpy().flatten()
y_pred = y_test_pred.numpy().flatten()

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"MAE:  {mae}")
print(f"MSE:  {mse}")
print(f"RMSE: {rmse}")
print(f"R2:   {r2}")


EVALUATION
MAE:  2.3806
MSE:  7.6693
RMSE: 2.7694
R2:   0.8555


In [74]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

In [75]:
from torchviz import make_dot
import os

os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

sample_input = X_train_tensor[:1]
y_sample = model(sample_input)

dot = make_dot(y_sample, params=dict(model.named_parameters()))
dot.render("computation_graph", format="png", cleanup=True)
print("Graph saved as computation_graph.png")

Graph saved as computation_graph.png
